In [1]:
!pip install -q pytorch-lightning datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.8 MB/s eta 0:00:00


In [2]:
# Instalacion de librerias
import random
import torch
import numpy as np
import os
from pytorch_lightning import seed_everything
import matplotlib.pyplot as plt
import seaborn as sns
import re

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)# Store the average loss after eachepoch so we can plot them.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["TF_DETERMINISTIC_OPS"] = "1" # See:https://github.com/NVIDIA/tensorflow-determinism#confirmed-current-gpu-specific-sources-of-non-determinism-with-solutions
seed_everything(42, workers=True)

from datasets import Dataset, DatasetDict #, load_metric EN PRINCIPIO ESTÁ DESCONTINUADO, TENDREMOS QUE BUSCAR OTRA ALTERNATIVA
import pandas as pd
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from huggingface_hub import login

INFO:lightning_fabric.utilities.seed:Seed set to 42


In [3]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

✅ GPU detectada. Trabajando con: "Tesla T4"
Memoria VRAM total disponible: 15.64 GB


Montura de google drive para poder acceder a los datasets

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Lectura y etiquetado de Datos

In [5]:
# ================================
# 1. Carga y preparación de datos
# ================================
# Cargar dataset de entrenamiento con etiquetas
#df = pd.read_csv("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/EXIST 2025 Videos Dataset/training/EXIST2025_training_task3_1_DISCARD.csv")
df = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv")
df = df.rename(columns={"label_task_3_1_merged": "label"})
df = df.drop(columns=["Unnamed: 0"])
df["label"] = df["label"].astype(int)

# from sklearn.model_selection import train_test_split
# train_df, eval_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(eval_df)

# PENDIENTE DE DEFINIR UN FICHERO TRAIN, VAL Y TEST
# División estratificada
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print("Distribución fichero de entrenamiento:")
print(train_df.value_counts('label'))

print("Distribución fichero de test:")
print(test_df.value_counts('label'))

print("Distribución fichero de validación:")
print(val_df.value_counts('label'))


Distribución fichero de entrenamiento:
label
0    914
1    841
Name: count, dtype: int64
Distribución fichero de test:
label
0    196
1    181
Name: count, dtype: int64
Distribución fichero de validación:
label
0    196
1    180
Name: count, dtype: int64


# Preprocesado del texto

In [6]:
# Para ver el vocabulario del dataset

import pandas as pd

# Suponiendo que la columna de texto se llama 'text'
# Tokeniza y obtiene todas las palabras
vocabulario = set(
    palabra.lower()
    for fila in df["text"].dropna()
    for palabra in fila.split()
)

# Mostrar todo el vocabulario
print(vocabulario)

# (Opcional) Ver el número de palabras únicas
print(f"\nTotal de palabras únicas: {len(vocabulario)}")


{'poseídas', 'fake', 'cg', '303', 'killer.', "ares',", 'anduviera', 'herencia.', 'comocj6', 'gobierno,', 'wider', 'intendencia', 'x7i>', 'parodia', 'arianaiasmine,', 'atacada', 'half', 'time!', 'valiente,', 'limita', 'breast', 'hated', 'seriously', 'hunters', 'compartirles', 'discusión', 'qarzs', 'lobbed', 'negativamente', 'marzo', 'restrain', 'jajaja!', 'diario.', 'margarita', 'niñez.', 'underestimate', 'entendiendo.', '"je', 'tenoch', 'techniques', 'behavior.', 'empresa', '#fyp.', 'cunt', 'dios.que', 'compártelo.', 'newspaper', 'annoy', 'olivia', '6*empiezo', 'sabes,', 'hypothetically,', 'heartbeat,', 'vegí26462', 'terrible.', 'continuously', '"gaslighting"', 'impactado.', 'maybe', 'ian', 'chaves', 'version', 'cibeles.', 'pump.', 'squint,', 'makers', 'pierda', 'guidelines', 'poou', 'poquito', 'ofreció', 'imply', 'lulú', 'tose', 'paper', 'protesta.', 'ciertos', 'okay,', 'sin9*', 'barriguitas', 'humanitaria.', 'motos', 'descontracturando', 'contladroaa.', 'tenerme', 'coyayuv', 'yvb', '

In [7]:
import re

def remove_links(tweet):
    """Takes a string and removes web links from it"""
    tweet = re.sub(r'http\S+', '', tweet)        # remove http links
    tweet = re.sub(r'bit.ly/\S+', '', tweet)     # remove bitly links
    tweet = re.sub(r'\[link\]', '', tweet )      # remove [link]
    tweet = re.sub(r'\[url\]', '', tweet )       # remove [url]
    tweet = re.sub(r'pic.twitter\S+','', tweet)
    return tweet

def remove_users(tweet):
    """Takes a string and removes retweet and @user information"""
    tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet
    tweet = re.sub('(@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove tweeted at
    tweet = re.sub(r'\[user\]', '', tweet )                      # remove [user]
    return tweet

def remove_hashtags(tweet):
    """Takes a string and removes any hash tags"""
    tweet = re.sub('(#[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove hash tags
    return tweet

def remove_av(tweet):
    """Takes a string and removes AUDIO/VIDEO tags or labels"""
    tweet = re.sub('VIDEO:', '', tweet)  # remove 'VIDEO:' from start of tweet
    tweet = re.sub('AUDIO:', '', tweet)  # remove 'AUDIO:' from start of tweet
    return tweet

def remove_emojis(tweet):
    emoj = re.compile("["
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U0001F300-\U0001F5FF"  # Miscellaneous Symbols And Pictographs
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00010000-\U0010FFFF"
        u"\U0001F680-\U0001F6FF"  # Transport and Map Symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\ufe0f"  # dingbats

                      "]+", re.UNICODE)
    return re.sub(emoj, '', tweet)

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1453/1710960392.py:14: SyntaxWarning: invalid escape sequence '\s'
  tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet


Convertimos todo a minúsculas

In [8]:
campo_texto = 'text'

train_df[campo_texto] = train_df[campo_texto].str.lower()
val_df[campo_texto] = val_df[campo_texto].str.lower()
test_df[campo_texto] = test_df[campo_texto].str.lower()

# Definición de métricas

In [9]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

# Entrenamiento del modelo

In [10]:
# Cargamos el token de HuggingFace que lo tenemos en un fichero oculto e iniciamos sesión

#with open("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/huggingfaceToken.txt", "r") as file:
#    hf_token = file.read().strip()

from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)
print("Login completado con éxito")

# ruta_archivo = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/token_hf.txt'

# try:
#     # Abrimos el archivo en modo lectura ('r)
#     with open(ruta_archivo, 'r') as file:
#         hf_token = file.read().strip()

#         # Hacemos el login
#         login(token=hf_token)
#         print("Login completado con éxito")
# except:
#     print("Error: No se ha encontrado el archivo " + ruta_archivo + " en la ruta especificada")

Login completado con éxito


In [11]:
# Elección del modelo

model_checkpoint = 'FacebookAI/xlm-roberta-base'

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [12]:
# Se carga el modelo preentrenado
n_labels = 2

# El uso de una función de inicialización facilita la repetición del entrenamiento
# Se puede usar la misma función de inicialización en diferentes ejecuciones del código o en configuraciones de entrenamiento diferentes
# Esto facilita la repetición del entrenamiento y la reproducibilidad, ya que se puede inicializar el modelo
# de la misma manera en cada ejecución.

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint,
                                                              num_labels = n_labels) #, return_dict = True )
                                                              # use_auth_token = 'token propio de HugginFace')

In [13]:
# Para saber el nombre del modelo
model_name = model_checkpoint.split("/")[-1]
model_name

'xlm-roberta-base'

# Fine-tuning

In [14]:
# Selección de hiperparámetros
#BATCH_SIZE = 32
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 15
#LEARNING_RATE = 3e-5
LEARNING_RATE = 1e-5
MAX_LENGTH = 128
WEIGHT_DECAY = 0.1

In [15]:
def tokenize_data(examples):
  return tokenizer(examples[campo_texto], truncation=True, max_length=MAX_LENGTH, padding=True)

In [16]:
from transformers import Trainer, TrainingArguments

optim = ["adamw_hf", "adamw_torch", "adamw_apex_fused", "adafactor", "adamw_torch_xla"]

training_args = TrainingArguments(
    output_dir = 'results',
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1', # Cambiar la metrica por la que queremos ajustar
    weight_decay = WEIGHT_DECAY,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    save_total_limit = 3,
    optim = optim[1],
    push_to_hub = False,
    greater_is_better = True,
    logging_strategy = 'epoch'
)

#output_dir = '/home/adrian/Escritorio/DeepSexist/TrainingBooks/Task3.1/Research/ModelosTransformers'
output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Transformers/' + model_name

# Entrenamiento de cada modelo por separado
model_output_dir = f"{output_dir}/modelo_{model_checkpoint}"

# Convierte directamente los DataFrames de Pandas a Datasets de Hugging Face
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

# Reseteamos el formato para que no haya fallos
train_dataset.reset_format()
valid_dataset.reset_format()

# Obtenemos los nombres de todas las columnas originales
columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names

#Protegemos la columna "label" para no eliminarla accidentalmente

columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train:
    columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid:
    columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

# Redefinimos model_init aquí para aplicar la congelación (Layer Freezing)
# def model_init():
#     # 1. Cargamos el modelo base
#     model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)
#     # 2. Congelamos todas las capas base (el "cerebro")
#     for param in model.base_model.parameters():
#         param.requires_grad = False
#     # El classifier sigue estando activo por defecto
#     return model

# Descongelación parcial de las últimas 3 capas

# def model_init():
#     # 1. Cargamos el modelo base
#     model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)

#     # 2. Congelamos TODO el cerebro base por defecto
#     for param in model.base_model.parameters():
#         param.requires_grad = False

#     # 3. Buscamos dónde guarda el modelo sus capas (esto varía según el modelo que estemos trabajando)
#     if hasattr(model.base_model, 'encoder') and hasattr(model.base_model.encoder, 'layer'):
#         capas_transformers = model.base_model.encoder.layer
#     elif hasattr(model.base_model, 'layers'):
#         capas_transformers = model.base_model.layers
#     else:
#         print("No se encontró la estructura de capas estándar. Se entrenará solo con la cabeza.")
#         capas_transformers=[]

#     # 4. Descongelamos solo las últimas 3 capas para que se adapten al vocabulario (TikTok)
#     capas_a_descongelar = 3
#     if len(capas_transformers) >= capas_a_descongelar:
#         print("Descongelando las últimas " + str(capas_a_descongelar) + " capas de un total de " + str(len(capas_transformers)))
#         for capa in capas_transformers[-capas_a_descongelar:]:
#             for param in capa.parameters():
#                 param.requires_grad = True
#     return model

# Inicializa el entrenador
trainer = Trainer(
    model_init = model_init,
    args=training_args,
    compute_metrics = compute_metrics,
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    processing_class=tokenizer,
)

# Entrena el modelo
#trainer.train()

# Guarda el modelo entrenado
#trainer.save_model(model_output_dir)

Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Optimización de hiperparámetros

Esto lo hacemos porque ha sido el modelo transformers de entre multilingüe, español e inglés que mejor nos ha funcionado.

In [17]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 15.6 MB/s eta 0:00:00


In [18]:
import optuna
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback

model_checkpoint = 'xlm-roberta-base'
n_labels = 2

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)

In [19]:
def optuna_hp_space(trial):
    return {
        # Busca un Learning Rate entre 1e-5 y 5e-5 (escala logarítmica)
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),

        # Penalización para no memorizar (entre 0.01 y 0.1)
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),

        # Calentamiento del motor (entre el 0% y el 10% de los datos)
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.1),

        # Tamaño de batch (que elija el que mejor generalice)
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    }

In [20]:
training_args = TrainingArguments(
    output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Transformers/Optuna/xlm-roberta-base',
    num_train_epochs = 5,           # Lo capamos a 5 épocas por intento para que la búsqueda sea rápida
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',   # Apuntamos a tu métrica
    greater_is_better = True,
    fp16 = True,                    # Compresión activada para que Colab vuele
    push_to_hub = False,
    logging_strategy = 'epoch',
    disable_tqdm = False            # Para ver la barrita de progreso
)

In [21]:
trainer = Trainer(
    model_init = model_init,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset = encoded_train_dataset,
    eval_dataset = encoded_valid_dataset,
    processing_class = tokenizer,
    # Tolerancia cero: si en 2 épocas no mejora, aborta el intento y pasa al siguiente
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
print("🔍 Iniciando rastreo bayesiano de hiperparámetros con Optuna...")

best_trial = trainer.hyperparameter_search(
    direction = "maximize",                    # Queremos el número más alto posible
    backend = "optuna",                        # Motor de búsqueda
    hp_space = optuna_hp_space,                # Nuestra función de límites
    n_trials = 10,                             # Va a hacer 10 intentos (aprox 1.5 - 2 horas)
    compute_objective = lambda metrics: metrics["eval_f1"] # El objetivo es maximizar tu eval_f1
)

[I 2026-06-25 14:37:11,892] A new study created in memory with name: no-name-005d84bb-b057-42dd-9369-4b7f87c7a1ae


🔍 Iniciando rastreo bayesiano de hiperparámetros con Optuna...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
1,0.687227,0.649897,0.659574,0.650060,0.667263,0.653741,0.653741,0.592357,0.707763,0.589965
2,0.630284,0.619534,0.683511,0.683006,0.682972,0.683050,0.683050,0.670360,0.695652,0.606301
3,0.569428,0.601509,0.696809,0.696594,0.696604,0.696939,0.696939,0.688525,0.704663,0.617811
4,0.500282,0.676651,0.667553,0.667551,0.668579,0.668651,0.668651,0.666667,0.668435,0.591433
5,0.407048,0.773162,0.680851,0.680272,0.680272,0.680272,0.680272,0.666667,0.693878,0.604019


...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
1,0.687025,0.620094,0.670213,0.669455,0.669561,0.669388,0.669388,0.653631,0.685279,0.594800
2,0.630326,0.602708,0.691489,0.691454,0.693636,0.693197,0.693197,0.694737,0.688172,0.611660
3,0.573064,0.588376,0.686170,0.686028,0.686170,0.686508,0.686508,0.679348,0.692708,0.608008


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-06-25 15:14:39,992] Trial 9 pruned. 


In [24]:
print("\n" + "="*50)
print("🚀 ¡BÚSQUEDA TERMINADA! 🚀")
print(f"🏆 El mejor F1 conseguido ha sido: {best_trial.objective}")
print(f"⚙️ Los hiperparámetros ganadores son:")
for key, value in best_trial.hyperparameters.items():
    print(f"   - {key}: {value}")
print("="*50)


🚀 ¡BÚSQUEDA TERMINADA! 🚀
🏆 El mejor F1 conseguido ha sido: 0.7091992762621067
⚙️ Los hiperparámetros ganadores son:
   - learning_rate: 1.8918391954099755e-05
   - weight_decay: 0.07357897539076494
   - warmup_ratio: 0.06608458824479
   - per_device_train_batch_size: 16


# Entrenamiendo del modelo después de la optimización de hiperparámetros

In [25]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification, EarlyStoppingCallback
from datasets import Dataset

In [26]:
# 1. Configuración del modelo
model_checkpoint = 'xlm-roberta-base'
n_labels = 2

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=n_labels)

In [27]:
# 2. Las Constantes Ganadoras de la Optimización con Optuna
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 10       # Le damos margen completo para buscar su verdadero pico
LEARNING_RATE = 1.8918e-05  # La tasa de aprendizaje perfecta descubierta
WEIGHT_DECAY = 0.07357      # Regularización fuerte anti-overfitting
WARMUP_RATIO = 0.06608      # Calentamiento lineal de gradientes

optim = ["adamw_hf", "adamw_torch", "adamw_apex_fused", "adafactor", "adamw_torch_xla"]

In [28]:
# 3. Rutas automáticas a tu Google Drive Principal
output_dir = '/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Transformers/XLM_RoBERTa_Optimizado'
model_output_dir = f"{output_dir}/modelo_final_tfm"

training_args = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    weight_decay = WEIGHT_DECAY,
    warmup_ratio = WARMUP_RATIO,     # <-- Inyectamos el Warmup ratio ganador
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    save_total_limit = 2,
    optim = optim[1],                # adamw_torch estable
    fp16 = True,                     # Compresión en GPU activa para máxima velocidad
    push_to_hub = False,
    greater_is_better = True,
    logging_strategy = 'epoch'
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [29]:
# 4. Preparación de Datasets (Conversión limpia de tus DataFrames de Pandas)
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names

columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train:
    columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid:
    columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

Map:   0%|          | 0/1755 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

In [30]:
# 5. Inicialización del Entrenador de Producción
trainer = Trainer(
    model_init = model_init,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset = encoded_train_dataset,
    eval_dataset = encoded_valid_dataset,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)], # Margen de 3 épocas sin mejorar para parar
    processing_class = tokenizer,
)

print("Lanzando el entrenamiento definitivo de XLM-RoBERTa con los mejores hiperparámetros...")
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Lanzando el entrenamiento definitivo de XLM-RoBERTa con los mejores hiperparámetros...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=990, training_loss=0.4103913413153754, metrics={'train_runtime': 607.4805, 'train_samples_per_second': 28.89, 'train_steps_per_second': 1.811, 'total_flos': 1038959779852800.0, 'train_loss': 0.4103913413153754, 'epoch': 9.0})

In [31]:
# 6. Guardado de seguridad eterno en tu Google Drive
print("Asegurando los pesos óptimos del modelo en la nube...")
trainer.save_model(model_output_dir)
print("¡Operación completada! El mejor modelo está a salvo en mi Drive.")

Asegurando los pesos óptimos del modelo en la nube...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

¡Operación completada! El mejor modelo está a salvo en mi Drive.
